In [1]:
from astroquery.heasarc import Heasarc
from astropy.coordinates import SkyCoord
import astropy.units as u
from datetime import datetime
from astropy.time import Time
import time
import pandas as pd

# List of transients with RA/Dec in decimal degrees
obj = {"name": "AT2025almb", "ra": 0.497662, "dec": 12.800505, "date": "2024-12-12"}


heasarc = Heasarc()
radius = 5 * u.arcmin  # Search radius
results_summary = []


print(f"Checking {obj['name']}...")
skypos = SkyCoord(ra=obj['ra']*u.deg, dec=obj['dec']*u.deg)
try:
	res = heasarc.query_region(skypos, radius=radius, catalog="swiftmastr")
except Exception as e:
	print(f"  Error querying {obj['name']}: {e}")
	res = None

if res is not None and len(res) > 0:
	has_uvot = res['uvot_exposure'] > 0
	matching = res[has_uvot]
	# Filter by date
	matching = matching[Time(matching['start_time'], format='mjd') > Time(obj["date"], format="iso")]


	if len(matching) > 0:
		print(f"  ✅ UVOT data found for {obj['name']} after given date (peak)")
		# Compute delay in days
		obs_times = Time(matching["start_time"], format="mjd")
		t_date = Time(obj["date"], format="iso")
		time_deltas = (obs_times - t_date).to_value("day")

		for row, delta, t_iso in zip(matching, time_deltas, obs_times.iso):
			results_summary.append({
				"name": obj["name"],
				"transient_ra": obj["ra"],
				"transient_dec": obj["dec"],
				"obs_ra": row["ra"],
				"obs_dec": row["dec"],
				"obsid": row["obsid"],
				"start_time_mjd": row["start_time"],
				"start_time_iso": t_iso,
				"exposure": row["uvot_exposure"],
				"days_since_peak": round(delta, 2),
			})
		
			# links = heasarc.locate_data(res)
			# heasarc.download_data(links, location = f"./swift_data/{row['obsid']}")

	else:
		print(f"  ❌ No UVOT observations in catalog entries for {obj['name']}")
else:
	print(f"  ❌ No Swift observations found for {obj['name']}")

# Save results
df = pd.DataFrame(results_summary)
# df.to_csv("uvot_matches.csv", index=False)
# print("\n✅ Saved UVOT-matched transients to 'uvot_matches.csv'")

Checking AT2025almb...
  ❌ No Swift observations found for AT2025almb
